# IBR-FaultScreen walkthrough

This notebook walks through one screening study on the IEEE 39-bus case:

1. Simulate a fault with and without an IBR
2. Label the result against the criteria (PRC-029-1 ride-through and others)
3. Plot the trajectories
4. Compute a critical clearing time
5. Look at the pre-fault features the classifier uses
6. Generate a small dataset, train and screen (the full workflow uses the command line)

It takes a few minutes to run. Results come from RMS simulation with generic IBR models:
a screening aid, not an interconnection study.

In [ ]:
import andes
andes.config_logger(stream_level=50)          # keep ANDES quiet

from ibr_faultscreen.scenario import Scenario, IBRSpec, FaultSpec
from ibr_faultscreen.simulate import simulate
from ibr_faultscreen.label import label
from ibr_faultscreen.criteria import default_criteria
from ibr_faultscreen.plots import plot_scenario

criteria = default_criteria()
print([e.name for e in criteria.envelopes])

## 1. Simulate

A bolted three-phase fault at bus 16, cleared in 6 cycles (0.1 s). In the second scenario,
synchronous generator 2 is replaced by a solar plant of the same rating and dispatch.

In [ ]:
base = Scenario("no_ibr", fault=FaultSpec(bus=16, clear_time=0.1))
with_ibr = Scenario("ibr_gen2", ibrs=(IBRSpec(2, "solar"),), fault=FaultSpec(bus=16, clear_time=0.1))

r0, r1 = simulate(base), simulate(with_ibr)
for r in (r0, r1):
    print(r.scenario.scenario_id, r.status, f"{r.runtime_s:.1f} s",
          f"faulted bus at {r.fault_residual_pu:.3f} pu during the fault")

## 2. Label

Each failure mode gets a flag: True (violated), False (passed) or None (not evaluated).

In [ ]:
for r in (r0, r1):
    lab = label(r, criteria)
    print(r.scenario.scenario_id, "RISKY" if lab.risky else "OK", lab.flags)
    for reason in lab.reasons:
        print("   ", reason[:140])

## 3. Plot

In [ ]:
from IPython.display import Image
path = plot_scenario(r1, label(r1, criteria), criteria, "walkthrough_ibr_gen2.png")
Image(filename=str(path), width=700)

## 4. Critical clearing time

Bisection to half a cycle. If bolted runs fail near the boundary, the result is a lower bound
plus the milder-fault CCT as an upper estimate.

In [ ]:
from ibr_faultscreen.cct import critical_clearing_time
cct = critical_clearing_time(Scenario("cct_b16", fault=FaultSpec(16)))
print(cct.text())

## 5. Features

Everything the classifier sees is known before the fault: SCR at each IBR, short-circuit level
at the faulted bus, electrical distances, synchronous inertia, clearing time.

In [ ]:
from ibr_faultscreen.features import compute_features
f = compute_features(with_ibr)
{k: round(v, 3) if isinstance(v, float) else v for k, v in f.items()}

## 6. Dataset, training and screening

A useful dataset has thousands of scenarios, so use the command line, which runs in parallel
and can resume:

```
ibr-faultscreen dataset --n 5000 --out dataset_output
ibr-faultscreen train dataset_output/dataset.csv --out model_output
ibr-faultscreen screen --model model_output/model.joblib --n 200 --out screen_output --open
```

Below is a tiny version (8 scenarios) just to show the files it writes.

In [ ]:
from ibr_faultscreen.dataset import SamplingPlan, sample_scenarios, run_dataset

if __name__ == "__main__":          # needed on Windows for the worker processes
    plan = SamplingPlan(n=8, seed=1, t_end=4.0)
    summary = run_dataset(sample_scenarios(plan), criteria, "walkthrough_dataset", workers=2,
                          plan=plan, progress=print)
    print(summary.counts)
    print(open(summary.data_card, encoding="utf-8").read()[:1200])